## Introduction to Hugging Face

Welcome to Harnessing Transformers with Hugging Face! This marks an exciting new chapter in your transformer journey. Over the previous three courses, you've built an extraordinary foundation: you explored the evolution from RNNs to attention mechanisms, deconstructed the entire Transformer architecture piece by piece, and mastered the art of training and inference with your own implementations. You've literally built transformers from scratch, understanding every mathematical detail and implementation nuance.

Now, we transition from building transformers to harnessing their power through the Hugging Face ecosystem. This shift represents a fundamental change in perspective: instead of implementing every component yourself, you'll learn to leverage the most powerful and widely used library in the NLP world. The deep understanding you've gained from building transformers from scratch will prove invaluable as we explore how Hugging Face abstracts these complexities while giving you the flexibility to customize and extend pre-trained models for your specific needs. In this first lesson, we'll explore the core abstractions that make Hugging Face so powerful: pipelines for high-level tasks and `Auto` classes for flexible model loading.

> **Note:** all code in this course run on CPU by default.

---

### The Hugging Face Ecosystem

The Hugging Face Transformers library represents one of the most significant democratization efforts in artificial intelligence. While you've mastered the intricate details of transformer architecture, the reality is that training state-of-the-art models requires enormous computational resources: models like GPT-3 cost millions of dollars to train, and even smaller models require extensive GPU clusters and weeks of training time. This creates a fundamental accessibility problem that limits who can participate in advancing NLP technology.

Hugging Face solves this accessibility problem by providing a comprehensive ecosystem built around pre-trained models. These models have already been trained on massive datasets using substantial computational resources, and they're made freely available for researchers, developers, and practitioners worldwide. The library offers three core abstractions that make working with these models incredibly straightforward: models (the neural network architectures), tokenizers (text preprocessing components), and pipelines (high-level interfaces for common tasks). This design philosophy means you can leverage cutting-edge NLP capabilities with just a few lines of code, while still maintaining the flexibility to customize and fine-tune models for your specific applications. The ecosystem includes thousands of pre-trained models, covering tasks from text classification to generation, and supporting over 100 languages.

---

### Working with Pipelines: High-Level NLP Made Simple

The pipeline API represents the highest-level interface in the Hugging Face ecosystem, designed to make common NLP tasks accessible with minimal code. A pipeline combines three essential components: a pre-trained model, its corresponding tokenizer, and the necessary pre- and post-processing logic. This abstraction allows you to perform complex tasks like sentiment analysis, text generation, or question answering with a single function call. Behind the scenes, the pipeline handles tokenization, model inference, and result formatting, allowing you to focus on your application logic rather than implementation details.

Let's explore this power through practical examples, starting with sentiment analysis:

```python
from transformers import pipeline

# Create a sentiment analysis pipeline with a specific model
sentiment_pipeline = pipeline("sentiment-analysis",
                              model="distilbert-base-uncased-finetuned-sst-2-english")

# Analyze sentiment of input text
result = sentiment_pipeline("I love this Transformer model!")
print(f"Sentiment: {result[0]['label']} (confidence: {result[0]['score']:.3f})")
```

This code creates a sentiment analysis pipeline using DistilBERT, a lightweight version of BERT that retains 97% of BERT's performance while being 60% smaller and 60% faster. The model has been specifically fine-tuned on the Stanford Sentiment Treebank (SST-2) dataset, which contains movie reviews labeled as positive or negative. When we call the pipeline with our input text, it returns a dictionary containing the predicted label and a confidence score. Running this code prints:

```text
Sentiment: POSITIVE (confidence: 1.000)
```

This shows perfect confidence for our clearly positive input.

---

### Text Generation and Question Answering Pipelines

Now, let's explore two more powerful pipeline types that demonstrate the versatility of this API:

```python
# Text Generation Pipeline - uses GPT-2 for autoregressive generation
generator = pipeline("text-generation", model="gpt2")
generated = generator("The future of AI is",
                      max_length=30,           # Limit output length
                      num_return_sequences=1,  # Generate one sequence
                      truncation=True)         # Handle long inputs gracefully
print(f"Generated: {generated[0]['generated_text']}")

# Question Answering Pipeline - extracts answers from context
qa_pipeline = pipeline("question-answering",
                       model="distilbert-base-cased-distilled-squad")
context = "The Transformer model was introduced in 2017 by researchers at Google Brain."
answer = qa_pipeline(question="When was the Transformer introduced?",
                     context=context)
print(f"Answer: {answer['answer']} (confidence: {answer['score']:.3f})")
```

The text generation pipeline uses GPT-2, a decoder-only transformer that excels at generating coherent text continuations. When we run it, we get output like:

```text
Generated: The future of AI is uncertain. Will this happen with social and environmental issues like the global warming crisis or not? We will do what we can to
```

Notice how the generated text maintains topical coherence while introducing related concepts.

The question answering pipeline uses a DistilBERT model fine-tuned on the SQuAD dataset, producing:

```text
Answer: 2017 (confidence: 0.969)
```

The high confidence score indicates the model found a clear answer in the provided context.

---

### Understanding Auto Classes: Flexible Model Loading

While pipelines provide convenient high-level access, the `Auto` classes offer more flexibility and control over model loading and usage. These classes (`AutoModel`, `AutoTokenizer`, and `AutoConfig`) automatically determine the correct model architecture, tokenizer type, and configuration based on the model name you provide. This abstraction solves a crucial problem: with hundreds of different model architectures available in the Hugging Face ecosystem, it would be impractical to manually specify the exact model class for each one. Instead, the `Auto` classes examine the model's configuration files and automatically instantiate the appropriate classes.

Let's explore how to use these `Auto` classes to load and work with pre-trained models directly:

```python
from transformers import AutoModel, AutoTokenizer, AutoConfig
import torch

# Specify the model we want to use
model_name = "distilbert-base-uncased"

# Load tokenizer, model, and configuration automatically
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

# Display model architecture information
print(f"Model: {model_name}")
print(f"Hidden size: {config.hidden_size}")
print(f"Number of layers: {config.num_hidden_layers}")
print(f"Number of attention heads: {config.num_attention_heads}")
```

The `from_pretrained()` method downloads and caches the model files on first use, then loads them into memory. The configuration object provides access to all architectural parameters, revealing that DistilBERT uses a hidden size of 768, 6 transformer layers, and 12 attention heads. This information helps us understand the model's capacity and computational requirements. Running the above snippet produces the following output:

```text
Exploring AutoModel and AutoTokenizer...
Model: distilbert-base-uncased
Hidden size: 768
Number of layers: 6
Number of attention heads: 12
```

> **Note:** When working with downstream tasks such as classification or question answering, you will typically use specialized `Auto` classes like `AutoModelForSequenceClassification` or `AutoModelForQuestionAnswering` instead of the base `AutoModel`. These classes automatically load the appropriate model heads for your task (e.g., a classification layer for sequence classification, or a span prediction head for question answering), making it easy to fine-tune or perform inference on task-specific datasets.

---

### Tokenization and Inference with Auto Classes

Now, let's see how tokenization and model inference work with these components:

```python
# Test tokenization - convert text to tokens and IDs
text = "Hello, how are you today?"
tokens = tokenizer.tokenize(text)  # Get individual tokens
token_ids = tokenizer.encode(text, return_tensors="pt")  # Get token IDs as PyTorch tensor

print(f"Tokens: {tokens}")
print(f"Token IDs shape: {token_ids.shape}")

# Perform model inference
with torch.no_grad():  # Disable gradient computation for efficiency
    outputs = model(token_ids)
    hidden_states = outputs.last_hidden_state

# Analyze model output and size
print(f"Output shape: {hidden_states.shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
```

Running this code produces the following output:

```text
Tokens: ['hello', ',', 'how', 'are', 'you', 'today', '?']
Token IDs shape: torch.Size([1, 9])
Output shape: torch.Size([1, 9, 768])
Model parameters: 66,362,880
```

The tokenizer converts our text into subword tokens: `['hello', ',', 'how', 'are', 'you', 'today', '?']`, then encodes these as numerical IDs that the model can process. The `return_tensors="pt"` argument formats the output as PyTorch tensors with shape `[1, 9]` (batch size 1, sequence length 9). When we pass these token IDs through the model, we get hidden states with shape `[1, 9, 768]`, where each of the 9 tokens is represented by a 768-dimensional vector encoding rich contextual information. The model contains over 66 million parameters, demonstrating the impressive scale of even "small" models like DistilBERT.

---

### Conclusion and Next Steps

Congratulations on taking your first steps into the Hugging Face ecosystem! You've successfully explored the fundamental abstractions that make this library so powerful: pipelines for high-level tasks with minimal code, and `Auto` classes for flexible model loading and customization. The transition from building transformers from scratch to leveraging pre-trained models represents a significant shift in how we approach NLP problems, moving from implementation details to application focus while maintaining the deep understanding you've developed.

Your journey through building transformers from the ground up provides invaluable insight into what's happening behind these convenient abstractions. This knowledge will serve you well as we continue exploring more advanced features of the Hugging Face ecosystem in upcoming lessons. The practice exercises that follow will give you hands-on experience with these core concepts, solidifying your understanding of pipelines and `Auto` classes before we dive into foundational Transformer architectures. Keep learning!


## Your First Sentiment Analysis Pipeline

Now that you've learned about the Hugging Face Transformers library and its powerful pipeline abstraction, it's time to put that knowledge into practice with your first hands-on exercise.

Your objective is to complete the sentiment analysis demo by creating a working pipeline that can classify text as positive or negative.

---

### Your Task

You'll need to:

- Set up a sentiment analysis pipeline using the specified pre-trained model;
- Apply the pipeline to analyze the provided test sentences;
- Print the label and the score of each sentence.

The provided code already includes the test sentences and result formatting — you just need to fill in the missing pipeline creation and usage. This is your first step toward understanding how simple it is to leverage pre-trained models for real NLP tasks!

---

### Starter Code

```python
from transformers import pipeline


def sentiment_analysis_demo():
    """Demonstrate sentiment analysis using Hugging Face pipeline"""
    print("Sentiment Analysis with Hugging Face Transformers")

    # TODO: Create a sentiment analysis pipeline using the "distilbert-base-uncased-finetuned-sst-2-english" model

    # Test sentences
    test_sentences = [
        "I love this Transformer model!",
        "This is terrible and I hate it.",
        "The weather is gloomy today.",
        "I am so excited about learning transformers!"
    ]

    # Analyze each sentence
    for sentence in test_sentences:
        # TODO: Use the pipeline to analyze the sentence

        # TODO: Print label and score


def main():
    sentiment_analysis_demo()


if __name__ == "__main__":
    main()
```


---

### Solution

Three things were missing: creating the pipeline, calling it on each sentence, and printing the result.

```python
# Create the pipeline once, before the loop
sentiment_pipeline = pipeline("sentiment-analysis",
                              model="distilbert-base-uncased-finetuned-sst-2-english")

for sentence in test_sentences:
    result = sentiment_pipeline(sentence)  # list with one dict: [{'label': ..., 'score': ...}]
    print(f"Sentence: {sentence}")
    print(f"Label: {result[0]['label']}, Score: {result[0]['score']:.3f}")
```

- `pipeline("sentiment-analysis", model=...)` bundles the tokenizer, the DistilBERT model fine-tuned on SST-2, and the post-processing into one callable.
- The pipeline is created **outside** the loop, so the model is loaded and cached once instead of once per sentence.
- Calling the pipeline with a single string returns a list with one dictionary, so the label and score are read with `result[0]['label']` and `result[0]['score']`.
- Without the pipeline creation, calling `sentiment_pipeline(...)` raises a `NameError`.
- Without indexing `[0]`, `result['label']` raises a `TypeError`, because `result` is a list.

When the script runs, each of the four test sentences is printed with a `POSITIVE` or `NEGATIVE` label and its confidence score. The first run downloads the model, so it needs internet access.


In [ ]:
from transformers import pipeline


def sentiment_analysis_demo():
    """Demonstrate sentiment analysis using Hugging Face pipeline"""
    print("Sentiment Analysis with Hugging Face Transformers")

    # Create a sentiment analysis pipeline using the "distilbert-base-uncased-finetuned-sst-2-english" model
    sentiment_pipeline = pipeline("sentiment-analysis",
                                  model="distilbert-base-uncased-finetuned-sst-2-english")

    # Test sentences
    test_sentences = [
        "I love this Transformer model!",
        "This is terrible and I hate it.",
        "The weather is gloomy today.",
        "I am so excited about learning transformers!"
    ]

    # Analyze each sentence
    for sentence in test_sentences:
        # Use the pipeline to analyze the sentence
        result = sentiment_pipeline(sentence)

        # Print label and score
        print(f"Sentence: {sentence}")
        print(f"Label: {result[0]['label']}, Score: {result[0]['score']:.3f}")


def main():
    sentiment_analysis_demo()


if __name__ == "__main__":
    main()


## Building Your First Text Generator

After exploring sentiment analysis in the previous exercise, you'll now dive into text generation with transformers. Your goal is to create a text generation pipeline that can produce creative continuations of text prompts.

---

### Your Task

The provided code has the structure ready, but you need to complete the missing parts:

- Create a text-generation pipeline using the GPT-2 model.
- Generate text with 2 different continuations and a `max_length` of 50 tokens.

Make sure to set the `truncation` parameter to `True` for both generations. Once you complete the code, you'll see how powerful pre-trained models can be at generating human-like text!

---

### Starter Code

```python
from transformers import pipeline


def text_generation_demo():
    """Demonstrate text generation using Hugging Face pipeline"""
    print("Text Generation with Hugging Face Transformers")

    # TODO: Create a text generation pipeline using the "gpt2" model

    # Generate text from a prompt
    prompt = "The future of artificial intelligence is"
    # TODO: Use the pipeline to generate text from the prompt
    # Set max_length=50, num_return_sequences=2, truncation=True

    print(f"Original prompt: {prompt}")
    print("\nGenerated text:")
    for i, result in enumerate(generated):
        print(f"{i+1}. {result['generated_text']}")


def main():
    text_generation_demo()


if __name__ == "__main__":
    main()
```


---

### Solution

Two things were missing: creating the text-generation pipeline, and calling it to define `generated`.

```python
# Create the text generation pipeline with GPT-2
generator = pipeline("text-generation", model="gpt2")

# Generate two continuations of the prompt
generated = generator(prompt,
                      max_length=50,
                      num_return_sequences=2,
                      truncation=True)
```

- `pipeline("text-generation", model="gpt2")` loads GPT-2 and its tokenizer behind one callable.
- `max_length=50` caps each output at 50 tokens, **including** the prompt tokens.
- `num_return_sequences=2` returns a list with two dictionaries, one per continuation, each with a `generated_text` key.
- `truncation=True` makes the pipeline cut inputs that are too long instead of raising an error.
- The result must be stored in `generated`, because the loop below it iterates over `generated`. Without it, the script raises a `NameError`.

When the script runs, it prints the original prompt followed by two numbered continuations. GPT-2 samples its output, so the text differs from run to run. The first run downloads the model, so it needs internet access.


In [ ]:
from transformers import pipeline


def text_generation_demo():
    """Demonstrate text generation using Hugging Face pipeline"""
    print("Text Generation with Hugging Face Transformers")

    # Create a text generation pipeline using the "gpt2" model
    generator = pipeline("text-generation", model="gpt2")

    # Generate text from a prompt
    prompt = "The future of artificial intelligence is"
    # Use the pipeline to generate text from the prompt
    # Set max_length=50, num_return_sequences=2, truncation=True
    generated = generator(prompt,
                          max_length=50,
                          num_return_sequences=2,
                          truncation=True)

    print(f"Original prompt: {prompt}")
    print("\nGenerated text:")
    for i, result in enumerate(generated):
        print(f"{i+1}. {result['generated_text']}")


def main():
    text_generation_demo()


if __name__ == "__main__":
    main()


## Building Question Answering Pipeline

Now that you've worked with sentiment analysis and text generation pipelines, let's explore another powerful capability: question answering. This exercise will show you how transformers can understand context and extract specific information from text.

---

### Your Task

Your goal is to complete the question answering section in the provided code. You'll need to:

- Create a question-answering pipeline using the specified model.
- Use it to answer a question about when the Transformer model was introduced.

The context text is already provided, and you'll be asking when the Transformer was introduced. Pay attention to how the pipeline returns both the answer and a confidence score, just as the sentiment analysis pipeline did.

This type of question answering is incredibly useful for building chatbots, search systems, and information extraction tools from documents.

---

### Starter Code

```python
import torch
from transformers import pipeline


def explore_pipelines():
    """Explore different Hugging Face pipelines"""
    print("Exploring Hugging Face Pipelines...")

    # Question Answering Pipeline
    # TODO: Create a question answering pipeline using "distilbert-base-cased-distilled-squad" model
    qa_pipeline = None
    context = "The Transformer model was introduced in 2017 by researchers at Google Brain."
    # TODO: Use the qa_pipeline to answer the question "When was the Transformer introduced?" with the given context
    answer = None
    print(f"Answer: {answer['answer']} (confidence: {answer['score']:.3f})")


def main():
    explore_pipelines()


if __name__ == "__main__":
    main()
```


---

### Solution

Two placeholders needed real values: `qa_pipeline` was `None`, and `answer` was `None`.

```python
# Create the question answering pipeline
qa_pipeline = pipeline("question-answering",
                       model="distilbert-base-cased-distilled-squad")
context = "The Transformer model was introduced in 2017 by researchers at Google Brain."

# Ask the question against the given context
answer = qa_pipeline(question="When was the Transformer introduced?",
                     context=context)
```

- `pipeline("question-answering", model=...)` loads a DistilBERT model fine-tuned on SQuAD, together with its tokenizer.
- The pipeline takes two keyword arguments: `question` (what to ask) and `context` (the text to search for the answer).
- The model extracts a span from `context` rather than generating new text.
- For a single question, the result is one dictionary, not a list, so `answer['answer']` and `answer['score']` work without `[0]`. This differs from the sentiment pipeline.
- Without the pipeline, `qa_pipeline` stays `None` and calling it raises a `TypeError`. Without the call, `answer['answer']` on `None` raises a `TypeError` too.

When the script runs, it should print the extracted answer, which should be `2017`, with its confidence score. The lesson above showed `2017 (confidence: 0.969)` for the same input. The first run downloads the model, so it needs internet access.


In [ ]:
import torch
from transformers import pipeline


def explore_pipelines():
    """Explore different Hugging Face pipelines"""
    print("Exploring Hugging Face Pipelines...")

    # Question Answering Pipeline
    # Create a question answering pipeline using "distilbert-base-cased-distilled-squad" model
    qa_pipeline = pipeline("question-answering",
                           model="distilbert-base-cased-distilled-squad")
    context = "The Transformer model was introduced in 2017 by researchers at Google Brain."
    # Use the qa_pipeline to answer the question "When was the Transformer introduced?" with the given context
    answer = qa_pipeline(question="When was the Transformer introduced?",
                         context=context)
    print(f"Answer: {answer['answer']} (confidence: {answer['score']:.3f})")


def main():
    explore_pipelines()


if __name__ == "__main__":
    main()


## Working with Core Transformer Components

After exploring high-level pipelines, it's time to understand the core components that power them. In this exercise, you'll work directly with `AutoModel`, `AutoTokenizer`, and `AutoConfig` to see how tokenization and model inference work behind the scenes.

---

### Your Task

Your goal is to complete the provided code by:

- Loading a tokenizer, model, and configuration from a pre-trained model;
- Tokenizing text and converting it to token IDs;
- Running the model to get hidden states from the input.

This hands-on experience will give you a deeper understanding of how transformers process text at a fundamental level, showing you exactly how text becomes numbers and flows through the model.

---

### Starter Code

```python
import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig


def explore_automodels():
    """Explore AutoModel and AutoTokenizer"""
    print("Exploring AutoModel and AutoTokenizer...")

    model_name = "distilbert-base-uncased"

    # TODO: Load tokenizer, model, and config using AutoTokenizer, AutoModel, and AutoConfig

    print(f"Model: {model_name}")
    print(f"Hidden size: {config.hidden_size}")
    print(f"Number of layers: {config.num_hidden_layers}")
    print(f"Number of attention heads: {config.num_attention_heads}")

    text = "Hello, how are you today?"
    # TODO: tokenize the text to get individual tokens

    # TODO: encode the text to get token IDs

    print(f"Tokens: {tokens}")
    print(f"Token IDs shape: {token_ids.shape}")

    # TODO: Perform model inference using torch.no_grad() and get hidden states

    print(f"Output shape: {hidden_states.shape}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    return model, tokenizer


def main():
    model, tokenizer = explore_automodels()


if __name__ == "__main__":
    main()
```


---

### Solution

Four TODOs needed code: loading the three components, tokenizing, encoding, and running inference.

```python
# Load tokenizer, model, and config
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

text = "Hello, how are you today?"
tokens = tokenizer.tokenize(text)                         # subword tokens (strings)
token_ids = tokenizer.encode(text, return_tensors="pt")   # token IDs as a PyTorch tensor

# Inference without gradient tracking
with torch.no_grad():
    outputs = model(token_ids)
    hidden_states = outputs.last_hidden_state
```

- `AutoTokenizer`, `AutoModel` and `AutoConfig` each read the model's config files and pick the matching class, so the same code works for other model names.
- `tokenizer.tokenize(text)` returns a list of token strings. `tokenizer.encode(text, return_tensors="pt")` returns the IDs as a tensor of shape `[1, sequence_length]`. The encoded sequence has two more entries than the token list because `encode` adds the special `[CLS]` and `[SEP]` tokens.
- `torch.no_grad()` turns off gradient tracking, which saves memory and time because we only run inference.
- `outputs.last_hidden_state` has shape `[batch_size, sequence_length, hidden_size]`, one 768-dimensional vector per token.
- Without the loading line, `config` is undefined and the first `print` raises a `NameError`. Without `tokenize` or `encode`, `tokens` or `token_ids` is undefined. Without the inference block, `hidden_states` is undefined.

When the script runs, it should print the model name, the config values (hidden size 768, 6 layers, 12 heads), the tokens, the shapes, and the parameter count. The lesson above showed `Token IDs shape: torch.Size([1, 9])`, `Output shape: torch.Size([1, 9, 768])` and `66,362,880` parameters for the same input. The first run downloads the model, so it needs internet access.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig


def explore_automodels():
    """Explore AutoModel and AutoTokenizer"""
    print("Exploring AutoModel and AutoTokenizer...")

    model_name = "distilbert-base-uncased"

    # Load tokenizer, model, and config using AutoTokenizer, AutoModel, and AutoConfig
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    config = AutoConfig.from_pretrained(model_name)

    print(f"Model: {model_name}")
    print(f"Hidden size: {config.hidden_size}")
    print(f"Number of layers: {config.num_hidden_layers}")
    print(f"Number of attention heads: {config.num_attention_heads}")

    text = "Hello, how are you today?"
    # tokenize the text to get individual tokens
    tokens = tokenizer.tokenize(text)

    # encode the text to get token IDs
    token_ids = tokenizer.encode(text, return_tensors="pt")

    print(f"Tokens: {tokens}")
    print(f"Token IDs shape: {token_ids.shape}")

    # Perform model inference using torch.no_grad() and get hidden states
    with torch.no_grad():
        outputs = model(token_ids)
        hidden_states = outputs.last_hidden_state

    print(f"Output shape: {hidden_states.shape}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    return model, tokenizer


def main():
    model, tokenizer = explore_automodels()


if __name__ == "__main__":
    main()
